In [24]:
!pip install mlflow dagshub

In [25]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.tensorflow

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

In [26]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("dagshub_token")

In [27]:
DAGSHUB_USERNAME = "Aryanupadhyay23"
DAGSHUB_REPO = "Emotion-Detection-Deep-Learning"

MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = secret_value_0

In [28]:
mlflow.set_experiment("RAF_DB_EfficientNetV2S")

mlflow.start_run(run_name="efficientnetv2s_mixup_cutmix")

mlflow.set_tags({
    "project":"Emotion-Detection-Deep-Learning",
    "github_user":"Aryanupadhyay23",
    "model":"EfficientNetV2S",
    "augmentation":"MixUp + CutMix",
    "framework":"TensorFlow"
})

In [29]:
DATASET_PATH = "/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
TEST_DIR  = os.path.join(DATASET_PATH, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50

EMOTIONS = ['Surprise','Fear','Disgust','Happy','Sad','Anger','Neutral']

mlflow.log_params({
    "model_name":"EfficientNetV2S",
    "image_size":IMG_SIZE,
    "batch_size":BATCH_SIZE,
    "epochs":EPOCHS,
    "augmentation":"mixup_cutmix"
})

In [30]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 12271 files belonging to 7 classes.
Found 3068 files belonging to 7 classes.


In [31]:
def apply_basic_aug(image, label):

    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_brightness(image, max_delta=0.1)

    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + 20, IMG_SIZE + 20)

    batch_size = tf.shape(image)[0]
    image = tf.image.random_crop(image, size=[batch_size, IMG_SIZE, IMG_SIZE, 3])

    return image, label

In [32]:
def sample_beta(alpha=0.2):
    """Samples from a Beta distribution using the Gamma distribution trick."""
    gamma_1 = tf.random.gamma(shape=[], alpha=alpha)
    gamma_2 = tf.random.gamma(shape=[], alpha=alpha)
    return gamma_1 / (gamma_1 + gamma_2)

def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lam = sample_beta(alpha)
    
    indices = tf.random.shuffle(tf.range(batch_size))
    images_shuffled = tf.gather(images, indices)
    labels_shuffled = tf.gather(labels, indices)
    
    images = lam * images + (1.0 - lam) * images_shuffled
    labels = lam * labels + (1.0 - lam) * labels_shuffled
    return images, labels

def cutmix(images, labels, alpha=1.0):
    batch_size = tf.shape(images)[0]
    image_size = tf.shape(images)[1]
    
    lam = sample_beta(alpha)
    
    # Calculate bounding box dimensions
    cut_rat = tf.math.sqrt(1. - lam)
    cut_w = tf.cast(tf.cast(image_size, tf.float32) * cut_rat, tf.int32)
    cut_h = tf.cast(tf.cast(image_size, tf.float32) * cut_rat, tf.int32)
    
    cx = tf.random.uniform([], minval=0, maxval=image_size, dtype=tf.int32)
    cy = tf.random.uniform([], minval=0, maxval=image_size, dtype=tf.int32)
    
    bbx1 = tf.clip_by_value(cx - cut_w // 2, 0, image_size)
    bby1 = tf.clip_by_value(cy - cut_h // 2, 0, image_size)
    bbx2 = tf.clip_by_value(cx + cut_w // 2, 0, image_size)
    bby2 = tf.clip_by_value(cy + cut_h // 2, 0, image_size)
    
    indices = tf.random.shuffle(tf.range(batch_size))
    images_shuffled = tf.gather(images, indices)
    labels_shuffled = tf.gather(labels, indices)
    
    # Create spatial mask
    xx, yy = tf.meshgrid(tf.range(image_size), tf.range(image_size))
    mask_x = (xx >= bbx1) & (xx < bbx2)
    mask_y = (yy >= bby1) & (yy < bby2)
    mask = tf.cast(mask_x & mask_y, tf.float32)
    mask = tf.expand_dims(mask, -1) 
    
    images = images * (1. - mask) + images_shuffled * mask
    
    # Adjust lambda to exact pixel area
    actual_lam = 1. - tf.cast((bbx2 - bbx1) * (bby2 - bby1), tf.float32) / tf.cast(image_size * image_size, tf.float32)
    labels = actual_lam * labels + (1. - actual_lam) * labels_shuffled
    
    return images, labels

def apply_advanced_aug(images, labels):
    """Randomly applies MixUp (40%), CutMix (40%), or nothing (20%)."""
    rand_val = tf.random.uniform([])
    
    # tf.cond expects callables returning tensors
    def apply_mixup(): return mixup(images, labels)
    def apply_cutmix(): return cutmix(images, labels)
    def apply_nothing(): return images, labels

    return tf.case(
        [
            (rand_val < 0.4, apply_mixup),
            ((rand_val >= 0.4) & (rand_val < 0.8), apply_cutmix),
        ],
        default=apply_nothing
    )

In [33]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(apply_basic_aug, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(apply_advanced_aug, num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [34]:
base_model = keras.applications.EfficientNetV2S(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_preprocessing=False
)

base_model.trainable = True

In [35]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = keras.ops.cast(inputs, "float32")
x = (x / 128.0) - 1.0

x = base_model(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(512, activation="swish")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(7, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_1 (Cast)                   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,995,943 (80.09 MB)

 Trainable params: 20,839,511 (79.50 MB)

 Non-trainable params: 156,432 (611.06 KB)

In [36]:
tf.keras.utils.plot_model(
    model,
    to_file="architecture.png",
    show_shapes=True
)

mlflow.log_artifact("architecture.png")

In [37]:
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

optimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy"]
)

mlflow.log_params({
    "optimizer":"Adam",
    "learning_rate":1e-4,
    "label_smoothing":0.1
})

In [38]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_efficientnetv2s_advanced.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),

    # Patience increased to 8 to allow model to learn through heavy regularization
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        verbose=1
    ),

    keras.callbacks.CSVLogger("training_log_advanced.csv")
]

In [39]:
start = time.time()

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

training_time = time.time() - start

mlflow.log_metric("training_time_seconds", training_time)

Epoch 1/50


I0000 00:00:1772854988.565573     154 service.cc:152] XLA service 0x7858e0002420 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772854988.565630     154 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1772855003.891051     154 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-07 03:43:45.679883: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:43:45.871562: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:43:46.339155: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accur

383/384 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.3462 - loss: 2.0407

2026-03-07 03:46:30.815209: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:46:31.003285: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:46:31.410935: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:46:31.607884: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-07 03:46:32.150560: E external/local_xla/xla/stream_

384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step - accuracy: 0.3464 - loss: 2.0402
Epoch 1: val_accuracy improved from -inf to 0.62647, saving model to best_efficientnetv2s_advanced.keras
384/384 ━━━━━━━━━━━━━━━━━━━━ 379s 528ms/step - accuracy: 0.3466 - loss: 2.0397 - val_accuracy: 0.6265 - val_loss: 1.2581 - learning_rate: 1.0000e-04
Epoch 2/50
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - accuracy: 0.6612 - loss: 1.2792
Epoch 3: val_accuracy improved from 0.73566 to 0.77966, saving model to best_efficientnetv2s_advanced.keras
384/384 ━━━━━━━━━━━━━━━━━━━━ 92s 239ms/step - accuracy: 0.6612 - loss: 1.2791 - val_accuracy: 0.7797 - val_loss: 0.9356 - learning_rate: 1.0000e-04
Epoch 4/50
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - accuracy: 0.7200 - loss: 1.1752
Epoch 4: val_accuracy improved from 0.77966 to 0.79954, saving model to best_efficientnetv2s_advanced.keras
384/384 ━━━━━━━━━━━━━━━━━━━━ 92s 240ms/step - accuracy: 0.7200 - loss: 1.1752 - val_accuracy: 0.7995 - val_loss: 0.8938 - learnin

In [40]:
for i in range(len(history.history["accuracy"])):

    mlflow.log_metric("train_accuracy", history.history["accuracy"][i], step=i)
    mlflow.log_metric("val_accuracy", history.history["val_accuracy"][i], step=i)
    mlflow.log_metric("train_loss", history.history["loss"][i], step=i)
    mlflow.log_metric("val_loss", history.history["val_loss"][i], step=i)

In [49]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend(loc="lower right")
plt.title("Training vs Validation Accuracy")
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend(loc="upper right")
plt.title("Training vs Validation Loss")
plt.grid(True)

plt.savefig("training_curves.png")
plt.close()

mlflow.log_artifact("training_curves.png")

In [50]:
y_true = []
y_pred = []
y_prob = []

for x, y in test_ds:

    preds = model.predict(x, verbose=0)

    y_true.extend(np.argmax(y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_prob = np.array(y_prob)

In [51]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=EMOTIONS,
    yticklabels=EMOTIONS
)

plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")

plt.savefig("confusion_matrix.png")
plt.close()

mlflow.log_artifact("confusion_matrix.png")

In [52]:
report = classification_report(y_true, y_pred, target_names=EMOTIONS)

print(report)

with open("classification_report.txt","w") as f:
    f.write(report)

mlflow.log_artifact("classification_report.txt")

              precision    recall  f1-score   support

    Surprise       0.85      0.84      0.84       329
        Fear       0.81      0.51      0.63        74
     Disgust       0.57      0.61      0.59       160
       Happy       0.95      0.94      0.94      1185
         Sad       0.86      0.81      0.83       478
       Anger       0.85      0.75      0.80       162
     Neutral       0.79      0.88      0.83       680

    accuracy                           0.86      3068
   macro avg       0.81      0.76      0.78      3068
weighted avg       0.86      0.86      0.86      3068



In [53]:
y_true_bin = label_binarize(y_true, classes=range(len(EMOTIONS)))

for i in range(len(EMOTIONS)):

    fpr, tpr, _ = roc_curve(y_true_bin[:,i], y_prob[:,i])
    roc_auc = auc(fpr, tpr)

    mlflow.log_metric(f"roc_auc_{EMOTIONS[i]}", roc_auc)

In [55]:
plt.figure(figsize=(8,6))

for i in range(len(EMOTIONS)):

    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)

    # log AUC value
    mlflow.log_metric(f"roc_auc_{EMOTIONS[i]}", roc_auc)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{EMOTIONS[i]} (AUC = {roc_auc:.3f})"
    )

plt.plot([0,1],[0,1],'k--',linewidth=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-Class ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)

plt.savefig("roc_auc_curve.png")
plt.close()

mlflow.log_artifact("roc_auc_curve.png")

In [54]:
best_model = tf.keras.models.load_model("best_efficientnetv2s_advanced.keras")

mlflow.tensorflow.log_model(best_model, "model")

2026/03/07 04:37:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 04:37:19 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


In [56]:
mlflow.end_run()

🏃 View run efficientnetv2s_mixup_cutmix at: https://dagshub.com/Aryanupadhyay23/Emotion-Detection-Deep-Learning.mlflow/#/experiments/13/runs/379de3a672544d8793c99917fa12a15c
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Emotion-Detection-Deep-Learning.mlflow/#/experiments/13
